# 06 — Retry Logic (`core/retry.py`)
Two utilities:
- **`@with_retry(max_retries, backoff_factor, exceptions)`** — decorator with exponential backoff (1s → 2s → 4s)
- **`retry_agent_call(agent.execute, request)`** — safe wrapper that returns `AgentResult(success=False)` on final failure instead of raising


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. @with_retry — Decorator

In [ ]:
from core.retry import with_retry
import time

attempt_log = []

@with_retry(max_retries=3, backoff_factor=0.01)  # tiny backoff for demo
def unstable_api():
    attempt_log.append(time.time())
    if len(attempt_log) < 3:
        raise ConnectionError("API timeout")
    return {"data": "success"}

result = unstable_api()
print("Result:", result)
print("Attempts:", len(attempt_log))
print("Backoff gaps (seconds):", [round(attempt_log[i+1]-attempt_log[i], 3) for i in range(len(attempt_log)-1)])

## 2. @with_retry — All Retries Fail

In [ ]:
attempt_log2 = []

@with_retry(max_retries=3, backoff_factor=0.01)
def always_fails():
    attempt_log2.append(1)
    raise RuntimeError("Permanently broken")

try:
    always_fails()
except RuntimeError as e:
    print(f"Raised after {len(attempt_log2)} attempts: {e}")

## 3. @with_retry — Selective Exception Types

In [ ]:
@with_retry(max_retries=3, backoff_factor=0.01, exceptions=(ConnectionError,))
def selective_retry():
    raise ValueError("Wrong type!")  # NOT in exceptions tuple

try:
    selective_retry()
except ValueError as e:
    print("ValueError raised immediately (not retried):", e)

## 4. retry_agent_call — Safe Agent Wrapper

In [ ]:
from core.retry import retry_agent_call
from core.base_agent import AgentRequest, AgentResult, BaseAgent

class UnreliableAgent(BaseAgent):
    name = "unreliable_agent"
    capabilities = []
    _count = 0

    def _execute(self, request):
        UnreliableAgent._count += 1
        raise RuntimeError(f"Service unavailable (attempt {UnreliableAgent._count})")

agent = UnreliableAgent()
request = AgentRequest(query="test query")

# retry_agent_call returns AgentResult(success=False), never raises
result = retry_agent_call(agent.execute, request, max_retries=3)

print("success :", result.success)  # False
print("error   :", result.error)
print("summary :", result.summary)
print("attempts:", UnreliableAgent._count)

## 5. retry_agent_call — Succeeds on 3rd Try

In [ ]:
class EventuallyWorkingAgent(BaseAgent):
    name = "eventual_agent"
    capabilities = []
    _count = 0

    def _execute(self, request):
        EventuallyWorkingAgent._count += 1
        if EventuallyWorkingAgent._count < 3:
            raise ConnectionError("Not ready yet")
        return AgentResult(agent_name=self.name, success=True,
                           summary="Finally worked!", confidence=0.9)

agent2 = EventuallyWorkingAgent()
result = retry_agent_call(agent2.execute, AgentRequest(query="test"), max_retries=3)

print("success :", result.success)
print("summary :", result.summary)
print("attempts:", EventuallyWorkingAgent._count)

## 6. Backoff Timing Visualization

In [ ]:
import math

def backoff_schedule(max_retries=3, backoff_factor=1.0):
    print(f"Retry schedule (factor={backoff_factor}):")
    for attempt in range(1, max_retries):
        wait = backoff_factor * (2 ** (attempt - 1))
        bar = "█" * int(wait * 10)
        print(f"  After attempt {attempt}: wait {wait:5.1f}s  {bar}")

backoff_schedule(max_retries=4, backoff_factor=1.0)
print()
backoff_schedule(max_retries=4, backoff_factor=0.5)